# Vesuvius Surface Detection — SuPreM SegResNet (Run 12)

3D segmentation of papyrus surfaces in CT scans of Herculaneum scrolls.

**Metric:** `0.30 * TopoScore + 0.35 * SurfaceDice@tau=2 + 0.35 * VOI_score`

**Changes from Run 9:**
- **`fit_flat_cos` schedule** — replaces `fit_one_cycle`. Key insight from Run 10b-v2:
  `one_cycle` warmup is harmful for pretrained models (destabilizes SuPreM features).
  `flat_cos` keeps LR flat for 75% then cosine anneals for 25%. No warmup.
- **50 epochs** (was 30) — Run 10b-v2 showed model was still improving when LR hit near-zero.
  More epochs lets the cosine annealing phase play out fully.
- **Correct thresholds** — T_LOW=0.35, T_HIGH=0.80 (proven best in eval_inference.py sweep).
- **Periodic checkpoints** — saves every 5 epochs for post-training eval sweep.

**Carried from Run 9:**
- Plain SegResNet (no attention gates, no deep supervision — proven architecture)
- Hardcoded LR=1e-5, discriminative: encoder=1e-7, decoder=1e-6, head=1e-5
- Foreground-biased patch sampling (FG_BIAS=0.5)
- 160^3 patches, stride=80
- Loss: 0.2*BCE + 0.2*Dice + 0.3*clDice + 0.3*Boundary
- MONAI SegResNet (4.7M params) with SuPreM pre-trained weights

## 1. Imports & Config

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import tifffile
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from fastai.learner import Learner
from fastai.callback.core import Callback
from fastai.callback.schedule import lr_find, fit_one_cycle, fit_flat_cos
from fastai.callback.tracker import SaveModelCallback, TrackerCallback
from fastai.callback.fp16 import MixedPrecision
from fastai.data.core import DataLoaders
from fastai.metrics import Metric
from monai.networks.nets import SegResNet
from scipy.ndimage import distance_transform_edt, label as scipy_label
from scipy.ndimage import binary_closing
from scipy.ndimage import zoom as scipy_zoom
from scipy.ndimage import generate_binary_structure, binary_propagation
from topometrics.leaderboard import compute_leaderboard_score
import random
import zipfile

# Config
ROOT = Path("/workspace/vesuvius-kaggle-competition")
TRAIN_IMG = ROOT / "data" / "train_images"
TRAIN_LBL = ROOT / "data" / "train_labels"
TEST_IMG = ROOT / "data" / "test_images"
CKPT_DIR = ROOT / "checkpoints"
CKPT_DIR.mkdir(exist_ok=True)
PRETRAINED_WEIGHTS = ROOT / "pretrained_weights" / "supervised_suprem_segresnet_2100.pth"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PATCH_SIZE = 160       # Train on 160^3 random crops
STRIDE = 80            # Sliding window stride (uniform 80-voxel overlap)
BATCH_SIZE = 2
NUM_WORKERS = 4
EPOCHS = 50            # Longer training (was 30) — flat_cos needs more epochs
SEED = 42
N_VAL_VOLUMES = 5      # Number of val volumes for competition metric eval
T_LOW = 0.35           # Hysteresis low threshold (proven best in eval sweep)
T_HIGH = 0.80          # Hysteresis high threshold (proven best — 0.85 was too aggressive)
METRIC_DOWNSAMPLE = 4  # Downsample factor for competition metric (320 -> 80)
CLOSING_Z_RADIUS = 2   # Anisotropic closing: z extent
CLOSING_XY_RADIUS = 1  # Anisotropic closing: xy extent
DUST_MIN_SIZE = 100    # Remove connected components smaller than this
USE_TTA = True         # Test-time augmentation (7-fold)
FG_BIAS = 0.5          # Probability of foreground-biased crop (vs random crop)
LR = 1e-5              # Hardcoded — proven in Run 9
SAVE_BEST_AFTER = 25   # Only start saving best model after this epoch (avoids early noise)

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)


class PeriodicSaveCallback(Callback):
    """Save model checkpoint every N epochs."""
    order = 61

    def __init__(self, every=5, fname="model"):
        self.every = every
        self.fname = fname

    def after_epoch(self):
        if (self.epoch + 1) % self.every == 0:
            self.learn.save(f"{self.fname}_ep{self.epoch+1}")
            print(f"  Periodic checkpoint saved: {self.fname}_ep{self.epoch+1}")


class DelayedSaveCallback(TrackerCallback):
    """Like SaveModelCallback but only starts monitoring after start_epoch.
    Avoids selecting early noisy comp_score peaks (the v9/v10b problem)."""
    order = 61

    def __init__(self, monitor='comp_score', comp=np.greater, fname='best', start_epoch=25):
        super().__init__(monitor=monitor, comp=comp)
        self.fname = fname
        self.start_epoch = start_epoch

    def after_epoch(self):
        if self.epoch < self.start_epoch:
            return
        super().after_epoch()
        if self.new_best:
            self.learn.save(self.fname)
            print(f"  Better model found at epoch {self.epoch} with {self.monitor} value: {self.best}")


print(f"Device: {DEVICE}")
print(f"CUDA: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"Patch size: {PATCH_SIZE}^3, Stride: {STRIDE}, Batch size: {BATCH_SIZE}")
print(f"Pretrained weights: {PRETRAINED_WEIGHTS.name}")
print(f"Hysteresis thresholds: T_low={T_LOW}, T_high={T_HIGH}")
print(f"Anisotropic closing: z_radius={CLOSING_Z_RADIUS}, xy_radius={CLOSING_XY_RADIUS}")
print(f"TTA: {'ON (7-fold)' if USE_TTA else 'OFF'}")
print(f"Foreground-biased sampling: FG_BIAS={FG_BIAS}")
print(f"LR: {LR:.1e} (hardcoded)")
print(f"Schedule: fit_flat_cos (75% flat, 25% cosine anneal — no warmup)")
print(f"Epochs: {EPOCHS}")
print(f"Best model selection: comp_score, starting at epoch {SAVE_BEST_AFTER}")
print(f"Competition metric: {N_VAL_VOLUMES} val volumes, downsample={METRIC_DOWNSAMPLE}x")

<frozen importlib._bootstrap_external>:1297: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.


Device: cuda
CUDA: NVIDIA GeForce RTX 5090
Patch size: 160^3, Stride: 80, Batch size: 2
Pretrained weights: supervised_suprem_segresnet_2100.pth
Hysteresis thresholds: T_low=0.35, T_high=0.8
Anisotropic closing: z_radius=2, xy_radius=1
TTA: ON (7-fold)
Foreground-biased sampling: FG_BIAS=0.5
LR: 1.0e-05 (hardcoded)
Schedule: fit_flat_cos (75% flat, 25% cosine anneal — no warmup)
Epochs: 50
Best model selection: comp_score, starting at epoch 25
Competition metric: 5 val volumes, downsample=4x


## 2. Data Exploration

In [2]:
train_df = pd.read_csv(ROOT / "data" / "train.csv")
print(f"Training samples: {len(train_df)}")
print(f"Unique scroll_ids: {train_df.scroll_id.nunique()}")
print()
print(train_df.scroll_id.value_counts())

Training samples: 806
Unique scroll_ids: 6

scroll_id
34117    382
35360    176
26010    130
26002     88
44430     17
53997     13
Name: count, dtype: int64


In [3]:
# Load a sample volume and label — pick one that exists on disk
sample_id = None
for sid in train_df.id:
    if (TRAIN_IMG / f"{sid}.tif").exists():
        sample_id = sid
        break

sample_img = tifffile.imread(TRAIN_IMG / f"{sample_id}.tif")
sample_lbl = tifffile.imread(TRAIN_LBL / f"{sample_id}.tif")

print(f"Sample ID: {sample_id}")
print(f"Image shape: {sample_img.shape}, dtype: {sample_img.dtype}")
print(f"Label shape: {sample_lbl.shape}, dtype: {sample_lbl.dtype}")
print(f"Label unique values: {np.unique(sample_lbl)}")

fg_pct = (sample_lbl == 1).sum() / sample_lbl.size * 100
bg_pct = (sample_lbl == 0).sum() / sample_lbl.size * 100
ign_pct = (sample_lbl == 2).sum() / sample_lbl.size * 100
print(f"\nLabel distribution: bg={bg_pct:.1f}%, fg={fg_pct:.1f}%, unlabeled={ign_pct:.1f}%")

Sample ID: 2290837
Image shape: (320, 320, 320), dtype: uint8
Label shape: (320, 320, 320), dtype: uint8
Label unique values: [0 1 2]

Label distribution: bg=37.3%, fg=4.9%, unlabeled=57.8%


In [4]:
# Visualize 3 orthogonal slices through the volume
mid = sample_img.shape[0] // 2

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Row 1: raw CT slices
axes[0, 0].imshow(sample_img[mid, :, :], cmap="gray")
axes[0, 0].set_title(f"Axial (z={mid})")
axes[0, 1].imshow(sample_img[:, mid, :], cmap="gray")
axes[0, 1].set_title(f"Coronal (y={mid})")
axes[0, 2].imshow(sample_img[:, :, mid], cmap="gray")
axes[0, 2].set_title(f"Sagittal (x={mid})")

# Row 2: label overlay (red = foreground, blue = unlabeled)
for i, (slc_img, slc_lbl, title) in enumerate([
    (sample_img[mid], sample_lbl[mid], f"Axial (z={mid})"),
    (sample_img[:, mid], sample_lbl[:, mid], f"Coronal (y={mid})"),
    (sample_img[:, :, mid], sample_lbl[:, :, mid], f"Sagittal (x={mid})"),
]):
    rgb = np.stack([slc_img]*3, axis=-1).astype(float) / 255.0
    rgb[slc_lbl == 1] = [1, 0, 0]  # foreground = red
    rgb[slc_lbl == 2] = [0, 0, 1]  # unlabeled = blue
    axes[1, i].imshow(rgb)
    axes[1, i].set_title(f"{title} + labels")

for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Dataset

In [5]:
def compute_signed_distance_map(label, mask):
    """
    Compute signed distance transform from ground truth boundary.
    Negative inside foreground, positive outside, zero at boundary.
    Normalized by max distance so values are roughly in [-1, 1].
    Only computed within the valid mask (label != 2).
    """
    fg = label.astype(bool)
    bg = ~fg

    # Distance from foreground boundary (inside -> negative, outside -> positive)
    dist_inside = distance_transform_edt(fg)
    dist_outside = distance_transform_edt(bg)

    signed_dist = dist_outside - dist_inside

    # Normalize by max absolute distance for stable loss scaling
    max_dist = max(np.abs(signed_dist).max(), 1.0)
    signed_dist = signed_dist / max_dist

    # Zero out unlabeled regions
    signed_dist = signed_dist * mask

    return signed_dist.astype(np.float32)


class VesuviusDataset(Dataset):
    """3D volume dataset with random patch cropping for training."""

    def __init__(self, ids, img_dir, lbl_dir, patch_size=128, augment=False, fg_bias=0.0):
        self.ids = ids
        self.img_dir = Path(img_dir)
        self.lbl_dir = Path(lbl_dir)
        self.patch_size = patch_size
        self.augment = augment
        self.fg_bias = fg_bias

    def __len__(self):
        return len(self.ids)

    def _random_crop(self, img, lbl, ps):
        """Extract a random ps^3 patch from the volume."""
        d, h, w = img.shape
        z = random.randint(0, d - ps)
        y = random.randint(0, h - ps)
        x = random.randint(0, w - ps)
        return img[z:z+ps, y:y+ps, x:x+ps], lbl[z:z+ps, y:y+ps, x:x+ps]

    def _fg_crop(self, img, lbl, ps):
        """Extract a ps^3 patch centered on a random foreground voxel."""
        fg_coords = np.argwhere(lbl == 1)
        if len(fg_coords) == 0:
            return self._random_crop(img, lbl, ps)

        # Pick a random foreground voxel
        idx = random.randint(0, len(fg_coords) - 1)
        cz, cy, cx = fg_coords[idx]

        # Center crop on it, clamped to volume bounds
        d, h, w = img.shape
        z = min(max(cz - ps // 2, 0), d - ps)
        y = min(max(cy - ps // 2, 0), h - ps)
        x = min(max(cx - ps // 2, 0), w - ps)
        return img[z:z+ps, y:y+ps, x:x+ps], lbl[z:z+ps, y:y+ps, x:x+ps]

    def __getitem__(self, idx):
        vol_id = self.ids[idx]

        # Load volume and label
        img = tifffile.imread(self.img_dir / f"{vol_id}.tif")
        lbl = tifffile.imread(self.lbl_dir / f"{vol_id}.tif")

        # Patch crop: foreground-biased or random
        if self.augment and random.random() < self.fg_bias:
            img, lbl = self._fg_crop(img, lbl, self.patch_size)
        else:
            img, lbl = self._random_crop(img, lbl, self.patch_size)

        # Normalize to [0, 1]
        img = img.astype(np.float32) / 255.0

        # Mask: True where label is 0 or 1 (valid), False where label is 2 (ignore)
        mask = (lbl != 2).astype(np.float32)

        # Binary label: foreground=1, everything else=0
        label = (lbl == 1).astype(np.float32)

        # Signed distance transform for boundary loss
        dist_map = compute_signed_distance_map(label, mask)

        # Augmentations: random flips and 90-degree rotations
        if self.augment:
            for axis in range(3):
                if random.random() > 0.5:
                    img = np.flip(img, axis=axis).copy()
                    label = np.flip(label, axis=axis).copy()
                    mask = np.flip(mask, axis=axis).copy()
                    dist_map = np.flip(dist_map, axis=axis).copy()

            k = random.randint(0, 3)
            plane_axes = random.choice([(0, 1), (0, 2), (1, 2)])
            if k > 0:
                img = np.rot90(img, k=k, axes=plane_axes).copy()
                label = np.rot90(label, k=k, axes=plane_axes).copy()
                mask = np.rot90(mask, k=k, axes=plane_axes).copy()
                dist_map = np.rot90(dist_map, k=k, axes=plane_axes).copy()

        # Add channel dimension: (1, D, H, W)
        img = torch.from_numpy(img).unsqueeze(0)
        label = torch.from_numpy(label).unsqueeze(0)
        mask = torch.from_numpy(mask).unsqueeze(0)
        dist_map = torch.from_numpy(dist_map).unsqueeze(0)

        return img, (label, mask, dist_map)


# Quick test
test_ds = VesuviusDataset([sample_id], TRAIN_IMG, TRAIN_LBL,
                          patch_size=PATCH_SIZE, augment=True, fg_bias=FG_BIAS)

# Run a few samples and check foreground stats
n_test = 10
fg_counts = []
for i in range(n_test):
    x, (y, m, d) = test_ds[0]
    fg_pct = y.sum().item() / y.numel() * 100
    fg_counts.append(fg_pct)

print(f"Image: {x.shape}, Label: {y.shape}, Mask: {m.shape}, DistMap: {d.shape}")
print(f"Image range: [{x.min():.2f}, {x.max():.2f}]")
print(f"DistMap range: [{d.min():.3f}, {d.max():.3f}]")
print(f"\nForeground-biased sampling test ({n_test} patches, fg_bias={FG_BIAS}):")
print(f"  FG% per patch: {['%.1f' % f for f in fg_counts]}")
print(f"  Mean FG%: {np.mean(fg_counts):.1f}%, patches with >1% FG: {sum(f > 1 for f in fg_counts)}/{n_test}")

Image: torch.Size([1, 160, 160, 160]), Label: torch.Size([1, 160, 160, 160]), Mask: torch.Size([1, 160, 160, 160]), DistMap: torch.Size([1, 160, 160, 160])
Image range: [0.00, 1.00]
DistMap range: [-0.047, 0.724]

Foreground-biased sampling test (10 patches, fg_bias=0.5):
  FG% per patch: ['8.2', '8.1', '8.2', '7.1', '8.0', '8.2', '6.1', '8.6', '7.8', '7.8']
  Mean FG%: 7.8%, patches with >1% FG: 10/10


## 4. DataLoaders

In [6]:
# Split by scroll_id: hold out scroll 26002 for validation (88 volumes)
train_df = pd.read_csv(ROOT / "data" / "train.csv")

# Only keep IDs that have both image and label files on disk
available_ids = set(
    int(p.stem) for p in TRAIN_IMG.glob("*.tif")
) & set(
    int(p.stem) for p in TRAIN_LBL.glob("*.tif")
)
train_df = train_df[train_df.id.isin(available_ids)].reset_index(drop=True)

VAL_SCROLL = 26002
val_ids = train_df[train_df.scroll_id == VAL_SCROLL].id.tolist()
trn_ids = train_df[train_df.scroll_id != VAL_SCROLL].id.tolist()

print(f"Train: {len(trn_ids)} volumes, Val: {len(val_ids)} volumes")

trn_ds = VesuviusDataset(trn_ids, TRAIN_IMG, TRAIN_LBL,
                          patch_size=PATCH_SIZE, augment=True, fg_bias=FG_BIAS)
val_ds = VesuviusDataset(val_ids, TRAIN_IMG, TRAIN_LBL,
                          patch_size=PATCH_SIZE, augment=False)

trn_dl = DataLoader(trn_ds, batch_size=BATCH_SIZE, shuffle=True,
                    num_workers=NUM_WORKERS, pin_memory=True)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=True)

dls = DataLoaders(trn_dl, val_dl, device=DEVICE)
print(f"DataLoaders ready. Training fg_bias={FG_BIAS}, Val fg_bias=0 (default).")

Train: 704 volumes, Val: 82 volumes
DataLoaders ready. Training fg_bias=0.5, Val fg_bias=0 (default).


## 5. SuPreM SegResNet Model

In [7]:
def create_segresnet_with_suprem_weights(weights_path, device="cpu"):
    """
    Create a MONAI SegResNet and load SuPreM supervised pre-trained weights.
    
    SuPreM was trained on 2,100 CT volumes with 25 organ classes.
    We load all layers except the final conv (which has out_channels=25
    in the checkpoint vs our out_channels=1).
    """
    # Match SuPreM's exact architecture config
    model = SegResNet(
        spatial_dims=3,
        in_channels=1,
        out_channels=1,        # binary segmentation (raw logits, no sigmoid)
        init_filters=16,       # channels: 16 -> 32 -> 64 -> 128
        blocks_down=[1, 2, 2, 4],
        blocks_up=[1, 1, 1],
        dropout_prob=0.2,      # add dropout for regularization
    )

    # Load SuPreM checkpoint
    ckpt = torch.load(weights_path, map_location=device, weights_only=False)
    pretrained = ckpt["net"]
    store_dict = model.state_dict()

    loaded, skipped = 0, 0
    for key in pretrained.keys():
        # Strip 'module.' prefix (SuPreM was trained with DataParallel)
        new_key = key.replace("module.", "")
        if new_key in store_dict:
            if "conv_final" in new_key:
                skipped += 1  # skip final conv (different out_channels)
                continue
            if store_dict[new_key].shape == pretrained[key].shape:
                store_dict[new_key] = pretrained[key]
                loaded += 1
            else:
                print(f"  Shape mismatch: {new_key}")
                skipped += 1
        else:
            skipped += 1

    model.load_state_dict(store_dict)
    print(f"SuPreM weights: loaded {loaded}/{loaded+skipped} params, skipped {skipped} (final conv)")
    return model


# Create model with pre-trained weights
model = create_segresnet_with_suprem_weights(PRETRAINED_WEIGHTS)

n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Model parameters: {n_params:.1f}M")

# Test forward pass
with torch.no_grad():
    dummy = torch.randn(1, 1, PATCH_SIZE, PATCH_SIZE, PATCH_SIZE)
    out = model(dummy)
    print(f"Input: {dummy.shape} -> Output: {out.shape}")

SuPreM weights: loaded 79/83 params, skipped 4 (final conv)
Model parameters: 4.7M


Input: torch.Size([1, 1, 160, 160, 160]) -> Output: torch.Size([1, 1, 160, 160, 160])


## 6. Loss Function

In [8]:
# --- clDice: topology-preserving loss via soft skeletons ---

def soft_erode_3d(img):
    """Soft erosion using min-pooling (negate, max-pool, negate)."""
    return -F.max_pool3d(-img, kernel_size=3, stride=1, padding=1)

def soft_dilate_3d(img):
    """Soft dilation using max-pooling."""
    return F.max_pool3d(img, kernel_size=3, stride=1, padding=1)

def soft_open_3d(img):
    """Soft morphological opening = erode then dilate."""
    return soft_dilate_3d(soft_erode_3d(img))

def soft_skeleton_3d(img, iters=5):
    """
    Compute soft skeleton via iterative erosion.
    skel = sum_i [ ReLU(erode^i(img) - open(erode^i(img))) ]
    """
    skel = F.relu(img - soft_open_3d(img))
    for _ in range(iters):
        img = soft_erode_3d(img)
        delta = F.relu(img - soft_open_3d(img))
        skel = skel + delta
    return skel

def soft_cldice(pred_sig, target, iters=5, smooth=1.0):
    """
    Compute clDice on 2x-downsampled volumes for memory efficiency.
    Topology is a coarse structural property, so 64^3 captures it well.
    """
    # Downsample 2x to reduce memory (128^3 -> 64^3)
    pred_ds = F.avg_pool3d(pred_sig, kernel_size=2)
    with torch.no_grad():
        target_ds = F.avg_pool3d(target, kernel_size=2)

    # Pred skeleton needs gradients for backprop
    skel_pred = soft_skeleton_3d(pred_ds, iters=iters)

    # Target skeleton does NOT need gradients
    with torch.no_grad():
        skel_target = soft_skeleton_3d(target_ds, iters=iters)

    # Topology precision: pred skeleton covered by target
    tprec = ((skel_pred * target_ds).sum() + smooth) / (skel_pred.sum() + smooth)
    # Topology sensitivity: target skeleton covered by pred
    tsens = ((skel_target * pred_ds).sum() + smooth) / (skel_target.sum() + smooth)

    cldice = 2.0 * tprec * tsens / (tprec + tsens + 1e-8)
    return cldice


# --- Boundary loss via signed distance transform ---

def boundary_loss(pred_sig, dist_map, mask):
    """
    Boundary loss: penalizes predictions far from the ground truth boundary.
    Uses signed distance transform (negative inside GT, positive outside).
    Minimizing mean(pred * dist_map) pushes predicted surface toward GT surface.

    Targets SurfaceDice@τ=2 component of competition metric.
    """
    # Multiply prediction probabilities by signed distance
    # - Predicting 1.0 where dist is positive (outside GT) → high penalty
    # - Predicting 1.0 where dist is negative (inside GT) → reward (negative loss)
    # - Predicting 0.0 anywhere → no contribution
    loss = (pred_sig * dist_map * mask).sum() / (mask.sum() + 1e-8)
    return loss


class MaskedCombinedLoss(nn.Module):
    """
    Combined loss: 0.2*BCE + 0.2*Dice + 0.3*clDice + 0.3*Boundary
    All masked to ignore unlabeled voxels (label=2).

    Each term targets a different aspect of the competition metric:
    - BCE: baseline voxel classification
    - Dice: volume overlap (class imbalance)
    - clDice: topology preservation → TopoScore (30%)
    - Boundary: surface accuracy → SurfaceDice@τ=2 (35%)
    """

    def __init__(self, bce_weight=0.2, dice_weight=0.2, cldice_weight=0.3,
                 boundary_weight=0.3, smooth=1.0, skel_iters=5):
        super().__init__()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.cldice_weight = cldice_weight
        self.boundary_weight = boundary_weight
        self.smooth = smooth
        self.skel_iters = skel_iters

    def forward(self, pred, target_tuple):
        target, mask, dist_map = target_tuple

        pred_sig = torch.sigmoid(pred)

        # Apply mask
        pred_masked = pred_sig * mask
        target_masked = target * mask

        # --- Masked BCE (on logits for numerical stability) ---
        bce = F.binary_cross_entropy_with_logits(pred, target, reduction='none')
        bce = (bce * mask).sum() / (mask.sum() + 1e-8)

        # --- Masked Dice ---
        intersection = (pred_masked * target_masked).sum()
        dice = (2.0 * intersection + self.smooth) / (
            pred_masked.sum() + target_masked.sum() + self.smooth
        )
        dice_loss = 1.0 - dice

        # --- Masked clDice (on downsampled patches) ---
        cldice = soft_cldice(pred_masked, target_masked,
                             iters=self.skel_iters, smooth=self.smooth)
        cldice_loss = 1.0 - cldice

        # --- Boundary loss ---
        bd_loss = boundary_loss(pred_sig, dist_map, mask)

        return (self.bce_weight * bce +
                self.dice_weight * dice_loss +
                self.cldice_weight * cldice_loss +
                self.boundary_weight * bd_loss)


class MaskedDiceMetric(Metric):
    """Dice coefficient metric (masked) for fastai Learner."""
    def __init__(self):
        self.reset()

    def reset(self):
        self.intersection = 0.0
        self.union = 0.0

    def accumulate(self, learn):
        pred = torch.sigmoid(learn.pred) > 0.5
        target, mask, _dist = learn.y
        pred_m = pred * mask
        tgt_m = target * mask
        self.intersection += (pred_m * tgt_m).sum().item()
        self.union += pred_m.sum().item() + tgt_m.sum().item()

    @property
    def value(self):
        return (2.0 * self.intersection + 1.0) / (self.union + 1.0)

    @property
    def name(self):
        return "dice"


# --- Sliding window inference (needed by CompetitionMetric during training) ---

def sliding_window_inference(model, volume, patch_size=160, stride=80, device="cuda"):
    """
    Run inference on a full 320^3 volume using overlapping patches.
    Overlapping regions are averaged for smoother predictions.
    """
    D, H, W = volume.shape
    ps = patch_size

    # Output accumulator and count (for averaging overlaps)
    output = np.zeros((D, H, W), dtype=np.float32)
    counts = np.zeros((D, H, W), dtype=np.float32)

    # Generate patch start positions
    def _positions(length, ps, stride):
        pos = list(range(0, length - ps, stride))
        if not pos or pos[-1] + ps < length:
            pos.append(length - ps)
        return pos

    z_pos = _positions(D, ps, stride)
    y_pos = _positions(H, ps, stride)
    x_pos = _positions(W, ps, stride)

    with torch.no_grad(), torch.amp.autocast("cuda"):
        for z in z_pos:
            for y in y_pos:
                for x in x_pos:
                    patch = volume[z:z+ps, y:y+ps, x:x+ps]
                    patch_t = torch.from_numpy(patch).float().unsqueeze(0).unsqueeze(0).to(device) / 255.0
                    logits = model(patch_t)
                    prob = torch.sigmoid(logits).squeeze().cpu().numpy()
                    output[z:z+ps, y:y+ps, x:x+ps] += prob
                    counts[z:z+ps, y:y+ps, x:x+ps] += 1.0

    # Average overlapping predictions
    output /= counts
    return output


# --- TTA: 7-fold test-time augmentation ---

def sliding_window_inference_tta(model, volume, patch_size=160, stride=80, device="cuda"):
    """
    7-fold TTA: original + 3 axis flips + 3 HW-plane rotations (90/180/270).
    Each augmentation is applied, inference is run, then the inverse is applied.
    All probability maps are averaged for the final result.
    """
    probs = []

    # 1. Original
    probs.append(sliding_window_inference(model, volume, patch_size, stride, device))

    # 2-4. Axis flips (z, y, x)
    for axis in range(3):
        flipped = np.flip(volume, axis=axis).copy()
        prob = sliding_window_inference(model, flipped, patch_size, stride, device)
        probs.append(np.flip(prob, axis=axis).copy())

    # 5-7. HW-plane (axes 1,2) rotations: 90, 180, 270 degrees
    for k in [1, 2, 3]:
        rotated = np.rot90(volume, k=k, axes=(1, 2)).copy()
        prob = sliding_window_inference(model, rotated, patch_size, stride, device)
        probs.append(np.rot90(prob, k=-k, axes=(1, 2)).copy())

    return np.mean(probs, axis=0)


# --- Hysteresis thresholding ---

def hysteresis_threshold(prob, t_low=0.45, t_high=0.85):
    """
    Dual-threshold seed-and-propagate.
    Strong seeds: prob >= t_high. Weak region: prob >= t_low.
    Seeds are propagated into the weak region using 26-connectivity.
    """
    strong = prob >= t_high
    weak = prob >= t_low
    struct = generate_binary_structure(3, 3)  # 26-connectivity
    return binary_propagation(strong, structure=struct, mask=weak).astype(np.uint8)


# --- Anisotropic closing ---

def build_anisotropic_struct(z_radius=2, xy_radius=1):
    """
    Build a z-heavy structuring element: disk-shaped in XY, extended in Z.
    Better for layered papyrus structure where surfaces are roughly planar.
    """
    sz = 2 * z_radius + 1
    sxy = 2 * xy_radius + 1
    struct = np.zeros((sz, sxy, sxy), dtype=bool)
    # For each z-slice, create a disk in XY
    cy, cx = xy_radius, xy_radius
    for y in range(sxy):
        for x in range(sxy):
            if (y - cy)**2 + (x - cx)**2 <= xy_radius**2:
                struct[:, y, x] = True
    return struct


# --- Competition metric: exact Kaggle scoring (downsampled for speed) ---

class CompetitionMetric(Metric):
    """
    Exact competition metric: 0.30*TopoScore + 0.35*SurfaceDice@τ=2 + 0.35*VOI
    
    Runs sliding window inference on N_VAL_VOLUMES validation volumes each epoch,
    downsamples pred+label by METRIC_DOWNSAMPLE (4x → 80^3) for fast Betti matching,
    computes the full leaderboard score via topometrics, returns the mean.
    
    Uses hysteresis thresholding instead of fixed threshold.
    """
    def __init__(self, val_ids, img_dir, lbl_dir, n_volumes=5, t_low=0.45,
                 t_high=0.85, patch_size=160, stride=80, downsample=4):
        self.val_ids = val_ids[:n_volumes]
        self.img_dir = Path(img_dir)
        self.lbl_dir = Path(lbl_dir)
        self.t_low = t_low
        self.t_high = t_high
        self.patch_size = patch_size
        self.stride = stride
        self.downsample = downsample
        self._model = None
        self.reset()

    def reset(self):
        self._model = None

    def accumulate(self, learn):
        # Just grab the model reference; actual computation happens in .value
        self._model = learn.model

    @property
    def value(self):
        if self._model is None:
            return 0.0

        self._model.eval()
        scores = []
        for vid in self.val_ids:
            img = tifffile.imread(self.img_dir / f"{vid}.tif")
            lbl = tifffile.imread(self.lbl_dir / f"{vid}.tif")

            prob = sliding_window_inference(
                self._model, img,
                patch_size=self.patch_size,
                stride=self.stride,
                device=next(self._model.parameters()).device,
            )
            pred = hysteresis_threshold(prob, self.t_low, self.t_high)

            # Downsample for fast Betti matching (~1s vs 20+ min at full res)
            ds = self.downsample
            if ds > 1:
                scale = 1.0 / ds
                pred = scipy_zoom(pred, scale, order=0).astype(np.uint8)
                lbl = scipy_zoom(lbl, scale, order=0).astype(np.uint8)

            report = compute_leaderboard_score(
                pred, lbl,
                ignore_label=2,
                spacing=(1, 1, 1),
                surface_tolerance=2.0,
                voi_alpha=0.3,
                combine_weights=(0.3, 0.35, 0.35),
            )
            scores.append(report.score)

        self._model.train()
        return float(np.mean(scores))

    @property
    def name(self):
        return "comp_score"


# --- Discriminative LR splitter for SegResNet ---

def segresnet_splitter(model):
    """Split SegResNet into 3 param groups for discriminative LR."""
    encoder = list(model.convInit.parameters()) + list(model.down_layers.parameters())
    decoder = list(model.up_layers.parameters()) + list(model.up_samples.parameters())
    head = list(model.conv_final.parameters())
    return [encoder, decoder, head]


print("Loss (0.2*BCE + 0.2*Dice + 0.3*clDice + 0.3*Boundary) and metrics defined.")
print("clDice → TopoScore, Boundary → SurfaceDice@τ=2")
print(f"CompetitionMetric: Kaggle score on {N_VAL_VOLUMES} val volumes, {METRIC_DOWNSAMPLE}x downsample")
print(f"  Hysteresis thresholding: T_low={T_LOW}, T_high={T_HIGH}")
print(f"  Sliding window: patch_size={PATCH_SIZE}, stride={STRIDE}")
print(f"TTA: 7-fold (original + 3 flips + 3 rotations) — used at inference only")
print(f"Splitter: 3 param groups (encoder/decoder/head) for discriminative LR")

Loss (0.2*BCE + 0.2*Dice + 0.3*clDice + 0.3*Boundary) and metrics defined.
clDice → TopoScore, Boundary → SurfaceDice@τ=2
CompetitionMetric: Kaggle score on 5 val volumes, 4x downsample
  Hysteresis thresholding: T_low=0.35, T_high=0.8
  Sliding window: patch_size=160, stride=80
TTA: 7-fold (original + 3 flips + 3 rotations) — used at inference only
Splitter: 3 param groups (encoder/decoder/head) for discriminative LR


## 7. Training

In [9]:
model = create_segresnet_with_suprem_weights(PRETRAINED_WEIGHTS, device=DEVICE)
model = model.to(DEVICE)
loss_fn = MaskedCombinedLoss()

comp_metric = CompetitionMetric(
    val_ids, TRAIN_IMG, TRAIN_LBL,
    n_volumes=N_VAL_VOLUMES, t_low=T_LOW, t_high=T_HIGH,
    patch_size=PATCH_SIZE, stride=STRIDE, downsample=METRIC_DOWNSAMPLE,
)

learn = Learner(
    dls,
    model,
    loss_func=loss_fn,
    metrics=[comp_metric],
    splitter=segresnet_splitter,
    cbs=[
        MixedPrecision(),
        # Only save best comp_score from epoch 25 onward (avoids early noise)
        DelayedSaveCallback(
            monitor="comp_score", comp=np.greater,
            fname="best_segresnet_v12", start_epoch=SAVE_BEST_AFTER,
        ),
        # Periodic checkpoints for post-training eval sweep
        PeriodicSaveCallback(every=5, fname="segresnet_v12"),
    ],
    path=CKPT_DIR,
)

print("Learner ready (3 param groups via segresnet_splitter).")
print(f"LR={LR:.1e} (hardcoded). Schedule: fit_flat_cos.")
print(f"Best model saved by comp_score starting at epoch {SAVE_BEST_AFTER}.")
print(f"Periodic checkpoints every 5 epochs.")

SuPreM weights: loaded 79/83 params, skipped 4 (final conv)
Learner ready (3 param groups via segresnet_splitter).
LR=1.0e-05 (hardcoded). Schedule: fit_flat_cos.
Best model saved by comp_score starting at epoch 25.
Periodic checkpoints every 5 epochs.


In [10]:
# Using hardcoded LR — same as Run 9 (proven effective)
print(f"Using hardcoded LR={LR:.1e}")
print(f"Discriminative LR: encoder={LR/100:.1e}, decoder={LR/10:.1e}, head={LR:.1e}")
print(f"Schedule: fit_flat_cos (75% flat, 25% cosine) — no warmup")

Using hardcoded LR=1.0e-05
Discriminative LR: encoder=1.0e-07, decoder=1.0e-06, head=1.0e-05
Schedule: fit_flat_cos (75% flat, 25% cosine) — no warmup


In [11]:
# fit_flat_cos: 75% flat LR, 25% cosine anneal to near-zero
# No warmup — avoids destabilizing pretrained SuPreM features (insight from Run 10b-v2)
print(f"Training {EPOCHS} epochs with fit_flat_cos, discriminative LR: slice({LR/100:.1e}, {LR:.1e})")
print(f"  Encoder (SuPreM):  ~{LR/100:.1e}")
print(f"  Decoder:           ~{LR/10:.1e}")
print(f"  Head (random init): {LR:.1e}")
print(f"  Schedule: flat for {int(EPOCHS*0.75)} epochs, cosine for {int(EPOCHS*0.25)} epochs")
print(f"  Best model selection: comp_score, epoch >= {SAVE_BEST_AFTER}")
learn.fit_flat_cos(EPOCHS, lr=slice(LR/100, LR))

Training 50 epochs with fit_flat_cos, discriminative LR: slice(1.0e-07, 1.0e-05)
  Encoder (SuPreM):  ~1.0e-07
  Decoder:           ~1.0e-06
  Head (random init): 1.0e-05
  Schedule: flat for 37 epochs, cosine for 12 epochs
  Best model selection: comp_score, epoch >= 25


epoch,train_loss,valid_loss,comp_score,time
0,0.511659,0.525501,0.559243,04:05
1,0.501889,0.512618,0.556246,04:04
2,0.496374,0.506668,0.562901,04:05
3,0.493711,0.494813,0.568258,04:06
4,0.491072,0.491917,0.573151,04:13
5,0.486545,0.490410,0.570154,04:11
6,0.489226,0.485988,0.574386,04:10
7,0.487778,0.485374,0.570265,04:13
8,0.486455,0.482697,0.576506,04:06
9,0.483678,0.481825,0.575011,04:09


  Periodic checkpoint saved: segresnet_v12_ep5


  Periodic checkpoint saved: segresnet_v12_ep10


  Periodic checkpoint saved: segresnet_v12_ep15


...

  Periodic checkpoint saved: segresnet_v12_ep30


  Periodic checkpoint saved: segresnet_v12_ep35


  Periodic checkpoint saved: segresnet_v12_ep40


  Periodic checkpoint saved: segresnet_v12_ep45


  Periodic checkpoint saved: segresnet_v12_ep50


In [12]:
# Plot training curves
learn.recorder.plot_loss()
plt.show()

## 8. Inference & Submission

In [13]:
# Load best model (selected by competition metric from second half of training)
learn.load("best_segresnet_v12")
model = learn.model.eval()
print("Best model loaded (selected by comp_score, epoch >= 25).")

Best model loaded (selected by comp_score, epoch >= 25).


In [14]:
# sliding_window_inference() and sliding_window_inference_tta() are defined in cell 14
print("Inference functions defined above:")
print(f"  sliding_window_inference — patch_size={PATCH_SIZE}, stride={STRIDE}")
print(f"  sliding_window_inference_tta — 7-fold TTA (original + 3 flips + 3 rotations)")
print(f"  TTA is {'ENABLED' if USE_TTA else 'DISABLED'} for final inference")

Inference functions defined above:
  sliding_window_inference — patch_size=160, stride=80
  sliding_window_inference_tta — 7-fold TTA (original + 3 flips + 3 rotations)
  TTA is ENABLED for final inference


### Hysteresis Parameter Sweep

Sweep T_low values with fixed T_high=0.85 to find optimal hysteresis thresholds.
Uses dice as a fast proxy, then verifies the best with comp_score on a few volumes.

In [15]:
# --- Post-processing pipeline: hysteresis + anisotropic closing + dust removal ---

def postprocess(probs, t_low=0.45, t_high=0.85, z_radius=2, xy_radius=1, min_size=100):
    """
    Full post-processing pipeline:
    1. Hysteresis thresholding (seed-and-propagate)
    2. Anisotropic morphological closing (z-heavy structuring element)
    3. Dust removal (remove small connected components)
    """
    # 1. Hysteresis thresholding
    binary = hysteresis_threshold(probs, t_low, t_high)

    # 2. Anisotropic closing
    struct = build_anisotropic_struct(z_radius, xy_radius)
    closed = binary_closing(binary, structure=struct)

    # 3. Dust removal — remove small connected components
    labeled, n_components = scipy_label(closed)
    if n_components > 0:
        component_sizes = np.bincount(labeled.ravel())
        small_mask = component_sizes < min_size
        small_mask[0] = False  # don't remove background
        closed[small_mask[labeled]] = 0

    return closed.astype(np.uint8)


def compute_dice_at_threshold(prob, label_vol, t_low, t_high):
    """Compute masked dice for a full volume using hysteresis thresholding."""
    mask = (label_vol != 2)
    target = (label_vol == 1)
    pred = hysteresis_threshold(prob, t_low, t_high).astype(bool)

    pred_m = pred & mask
    tgt_m = target & mask

    intersection = (pred_m & tgt_m).sum()
    union = pred_m.sum() + tgt_m.sum()
    return (2.0 * intersection + 1.0) / (union + 1.0)


# --- T_low sweep with fixed T_high ---
n_sweep = min(10, len(val_ids))
sweep_ids = val_ids[:n_sweep]
t_low_values = [0.35, 0.40, 0.45, 0.50, 0.55]
t_high_fixed = T_HIGH

print(f"Sweeping T_low in {t_low_values} with T_high={t_high_fixed} on {n_sweep} val volumes...")

# Pre-compute probability maps for all sweep volumes
prob_maps = {}
for i, vid in enumerate(sweep_ids):
    img = tifffile.imread(TRAIN_IMG / f"{vid}.tif")
    prob_maps[vid] = sliding_window_inference(model, img, patch_size=PATCH_SIZE, stride=STRIDE, device=DEVICE)
    print(f"  [{i+1}/{n_sweep}] {vid} — prob map computed")

# Sweep T_low using dice as fast proxy
results = {t: [] for t in t_low_values}
for vid in sweep_ids:
    lbl = tifffile.imread(TRAIN_LBL / f"{vid}.tif")
    prob = prob_maps[vid]
    for t in t_low_values:
        dice = compute_dice_at_threshold(prob, lbl, t, t_high_fixed)
        results[t].append(dice)

print(f"\n{'T_low':>8} {'T_high':>8} {'Dice (mean)':>12}")
print("-" * 30)
best_t_low, best_dice = T_LOW, 0
for t in t_low_values:
    mean_dice = np.mean(results[t])
    print(f"{t:>8.2f} {t_high_fixed:>8.2f} {mean_dice:>12.4f}")
    if mean_dice > best_dice:
        best_dice = mean_dice
        best_t_low = t

print(f"\nBest T_low by dice proxy: {best_t_low} (dice={best_dice:.4f})")

# Verify best with comp_score on 3 volumes
n_verify = min(3, n_sweep)
verify_scores = []
for vid in sweep_ids[:n_verify]:
    prob = prob_maps[vid]
    lbl = tifffile.imread(TRAIN_LBL / f"{vid}.tif")
    pred = postprocess(prob, t_low=best_t_low, t_high=t_high_fixed,
                       z_radius=CLOSING_Z_RADIUS, xy_radius=CLOSING_XY_RADIUS,
                       min_size=DUST_MIN_SIZE)

    ds = METRIC_DOWNSAMPLE
    pred_ds = scipy_zoom(pred, 1.0/ds, order=0).astype(np.uint8)
    lbl_ds = scipy_zoom(lbl, 1.0/ds, order=0).astype(np.uint8)

    report = compute_leaderboard_score(
        pred_ds, lbl_ds, ignore_label=2, spacing=(1,1,1),
        surface_tolerance=2.0, voi_alpha=0.3, combine_weights=(0.3, 0.35, 0.35),
    )
    verify_scores.append(report.score)
    print(f"  Verify {vid}: comp_score={report.score:.4f}")

print(f"\nVerification comp_score (mean of {n_verify}): {np.mean(verify_scores):.4f}")

BEST_T_LOW = best_t_low
BEST_T_HIGH = t_high_fixed
print(f"\nFinal thresholds: T_low={BEST_T_LOW}, T_high={BEST_T_HIGH}")

Sweeping T_low in [0.35, 0.4, 0.45, 0.5, 0.55] with T_high=0.8 on 10 val volumes...


  [1/10] 26894125 — prob map computed


  [2/10] 105796630 — prob map computed


  [3/10] 327851248 — prob map computed


  [4/10] 418613908 — prob map computed


...

  [10/10] 730065526 — prob map computed



   T_low   T_high  Dice (mean)
------------------------------
    0.35     0.80       0.2812
    0.40     0.80       0.2803
    0.45     0.80       0.2710
    0.50     0.80       0.2497
    0.55     0.80       0.2061

Best T_low by dice proxy: 0.35 (dice=0.2812)


  Verify 26894125: comp_score=0.5722


  Verify 105796630: comp_score=0.5414


  Verify 327851248: comp_score=0.5763

Verification comp_score (mean of 3): 0.5633

Final thresholds: T_low=0.35, T_high=0.8


In [16]:
# Run inference on test volumes with TTA + hysteresis + anisotropic closing + dust removal
import time

test_df = pd.read_csv(ROOT / "data" / "test.csv")
submission_dir = ROOT / "submission"
submission_dir.mkdir(exist_ok=True)

infer_fn = sliding_window_inference_tta if USE_TTA else sliding_window_inference
print(f"Inference mode: {'TTA (7-fold)' if USE_TTA else 'standard'}")
print(f"Post-processing: hysteresis(T_low={BEST_T_LOW}, T_high={BEST_T_HIGH}) + "
      f"aniso_closing(z={CLOSING_Z_RADIUS}, xy={CLOSING_XY_RADIUS}) + "
      f"dust_removal(min={DUST_MIN_SIZE})")

for _, row in test_df.iterrows():
    vol_id = row.id
    img_path = TEST_IMG / f"{vol_id}.tif"
    if not img_path.exists():
        print(f"Skipping {vol_id}: file not found")
        continue

    print(f"\nPredicting {vol_id}...")
    img = tifffile.imread(img_path)

    t0 = time.time()
    prob = infer_fn(model, img, patch_size=PATCH_SIZE, stride=STRIDE, device=DEVICE)
    t_infer = time.time() - t0

    pred = postprocess(prob, t_low=BEST_T_LOW, t_high=BEST_T_HIGH,
                       z_radius=CLOSING_Z_RADIUS, xy_radius=CLOSING_XY_RADIUS,
                       min_size=DUST_MIN_SIZE)

    out_path = submission_dir / f"{vol_id}.tif"
    tifffile.imwrite(out_path, pred)
    fg = pred.sum()
    print(f"  Inference: {t_infer:.1f}s")
    print(f"  Hysteresis: T_low={BEST_T_LOW}, T_high={BEST_T_HIGH}")
    print(f"  Saved {out_path} — shape: {pred.shape}, fg: {fg}/{pred.size} ({fg/pred.size*100:.1f}%)")

print("\nInference done.")

Inference mode: TTA (7-fold)
Post-processing: hysteresis(T_low=0.35, T_high=0.8) + aniso_closing(z=2, xy=1) + dust_removal(min=100)

Predicting 1407735...


  Inference: 11.9s
  Hysteresis: T_low=0.35, T_high=0.8
  Saved /workspace/vesuvius-kaggle-competition/submission/1407735.tif — shape: (320, 320, 320), fg: 0/32768000 (0.0%)

Inference done.


In [17]:
# Create submission.zip
zip_path = ROOT / "submission.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for tif in submission_dir.glob("*.tif"):
        zf.write(tif, tif.name)

print(f"Submission zip created: {zip_path}")
print(f"Size: {zip_path.stat().st_size / 1e6:.1f} MB")

Submission zip created: /workspace/vesuvius-kaggle-competition/submission.zip
Size: 0.0 MB


In [18]:
# Visualize a prediction (if test data is available)
if list(submission_dir.glob("*.tif")):
    pred_path = list(submission_dir.glob("*.tif"))[0]
    pred_vol = tifffile.imread(pred_path)
    mid = pred_vol.shape[0] // 2

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(pred_vol[mid], cmap="gray")
    axes[0].set_title(f"Axial z={mid}")
    axes[1].imshow(pred_vol[:, mid], cmap="gray")
    axes[1].set_title(f"Coronal y={mid}")
    axes[2].imshow(pred_vol[:, :, mid], cmap="gray")
    axes[2].set_title(f"Sagittal x={mid}")
    for ax in axes:
        ax.axis("off")
    plt.suptitle(f"Prediction: {pred_path.stem}")
    plt.tight_layout()
    plt.show()

In [19]:
# Trace the best model for Kaggle deployment (no MONAI dependency needed at inference)
print("Tracing best model for Kaggle...")
model.eval()
model_cpu = model.cpu()
dummy = torch.randn(1, 1, PATCH_SIZE, PATCH_SIZE, PATCH_SIZE)
traced = torch.jit.trace(model_cpu, dummy)

trace_path = ROOT / "kaggle" / "kaggle_weights_download" / "best_segresnet_v12_traced.pt"
traced.save(str(trace_path))
print(f"Traced model saved: {trace_path}")
print(f"Size: {trace_path.stat().st_size / 1e6:.1f} MB")

# Verify the trace
loaded = torch.jit.load(str(trace_path))
test_out = loaded(dummy)
print(f"Trace verification: input {dummy.shape} -> output {test_out.shape}")
print(f"Output range: [{test_out.min().item():.3f}, {test_out.max().item():.3f}]")
print("\nv12 training complete!")

Tracing best model for Kaggle...


Traced model saved: /workspace/vesuvius-kaggle-competition/kaggle/kaggle_weights_download/best_segresnet_v12_traced.pt
Size: 19.1 MB


Trace verification: input torch.Size([1, 1, 160, 160, 160]) -> output torch.Size([1, 1, 160, 160, 160])
Output range: [-3.507, 1.192]

v12 training complete!
